# 项目V1.0：多模型训练对比

**日期：** 2026-08-18 周二

## 今日目标

- 加载昨天预处理好的数据（或重新生成）
- 训练多个分类模型：随机森林、SVM、KNN、逻辑回归、决策树
- 用交叉验证对比各模型的准确率
- 选出表现最优的模型，为明天的 GridSearchCV 调参做准备

# 第一步：确保数据就绪

如果你昨天已经运行过预处理代码，且 `X_train_scaled`、`X_test_scaled`、`y_train`、`y_test` 还在内存中，可以直接跳到第二步。

如果内核重启过，重新运行下面的完整预处理代码：

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 读取数据
df = pd.read_csv("defect_data.csv")

# 标签编码
label_encoder = LabelEncoder()
df["严重程度_编码"] = label_encoder.fit_transform(df["严重程度"])

# OneHot编码特征
df_encoded = pd.get_dummies(
    df,
    columns=["缺陷类型", "复现概率"],
    prefix=["类型", "复现"],
    drop_first=True
)

# 分离 X 和 y
X = df_encoded.drop(["严重程度", "严重程度_编码"], axis=1)
y = df_encoded["严重程度_编码"]

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 标准化（用于 SVM、KNN、逻辑回归）
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("数据准备完成")
print(f"训练集形状: {X_train_scaled.shape}")
print(f"测试集形状: {X_test_scaled.shape}")
print(f"标签类别: {label_encoder.classes_.tolist()}")

数据准备完成
训练集形状: (350, 6)
测试集形状: (150, 6)
标签类别: ['一般', '严重', '致命', '轻微']


## 第二步：定义并训练多个模型

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {
    "随机森林":RandomForestClassifier(n_estimators = 100, max_depth = 5, random_state = 42),
    "SVM(RBF)":SVC(kernel = "rbf", gamma = "scale", random_state = 42),
    "KNN(K=5)":KNeighborsClassifier(n_neighbors = 5),
    "逻辑回归":LogisticRegression(max_iter = 1000, random_state = 42),
    "决策树":DecisionTreeClassifier(max_depth = 5, random_state = 42)
}

print(f"模型定义完成, 共 {len(models)} 个模型")

模型定义完成, 共 5 个模型


## 第三步：交叉验证对比

In [10]:
from sklearn.model_selection import cross_val_score

print("===== 5折交叉验证对比 =====")
results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv = 5, scoring = "accuracy")
    results[name] = scores.mean()
    print(f"{name}: 平均准确率 = {scores.mean():.3f}(±{scores.std():.3f})")

# 找出最优模型
best_name = max(results, key=results.get)
print(f"\n🏆 最优模型：{best_name}（交叉验证准确率 {results[best_name]:.3f}）")

===== 5折交叉验证对比 =====
随机森林: 平均准确率 = 0.906(±0.017)
SVM(RBF): 平均准确率 = 0.874(±0.035)
KNN(K=5): 平均准确率 = 0.840(±0.047)
逻辑回归: 平均准确率 = 0.954(±0.011)
决策树: 平均准确率 = 0.803(±0.060)

🏆 最优模型：逻辑回归（交叉验证准确率 0.954）


## 第四步：在测试集上评估最优模型

In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# 用最优模型重新训练
best_model = models[best_name]
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

print(f"\n====== 最优模型测试集评估 ======")
print(f"准确率: {accuracy_score(y_test, y_pred):.3f}")
print(f"\n分类报告:")
print(classification_report(y_test, y_pred, target_names = label_encoder.classes_.tolist()))
print(f"混淆矩阵: \n{confusion_matrix(y_test, y_pred)}")


====== 最优模型测试集评估 ======
准确率: 0.967

分类报告:
              precision    recall  f1-score   support

          一般       0.90      1.00      0.95        47
          严重       1.00      1.00      1.00        62
          致命       1.00      1.00      1.00        35
          轻微       1.00      0.17      0.29         6

    accuracy                           0.97       150
   macro avg       0.98      0.79      0.81       150
weighted avg       0.97      0.97      0.96       150

混淆矩阵: 
[[47  0  0  0]
 [ 0 62  0  0]
 [ 0  0 35  0]
 [ 5  0  0  1]]


## 今日验收标准
- 1. 成功训练 5 个模型并完成交叉验证对比

- 2. 输出每个模型的平均准确率和标准差

- 3. 选出最优模型，并在测试集上输出分类报告

## 2026-08-18 周二（第38天）项目V1.0：多模型训练对比

### 完成内容
- 加载并预处理缺陷数据（500条→350训练/150测试）
- 训练5个模型：随机森林、SVM、KNN、逻辑回归、决策树
- 5折交叉验证对比准确率
- 选出最优模型并在测试集评估

### 模型对比结果
| 模型 | 交叉验证准确率 | 标准差 |
|------|-------------|--------|
| 随机森林 | __0.906__ | __0.017__ |
| SVM(RBF) | __0.874__ | __0.035__ |
| KNN(K=5) | __0.840__ | __0.047__ |
| 逻辑回归 | __0.954__ | __0.011__ |
| 决策树 | __0.803__ | __0.060__ |

### 最优模型
- 名称：__逻辑回归__
- 测试集准确率：__0.967__
- 分类报告关键指标：
  - 宏平均F1：0.81
  - 加权平均F1：0.96
  - 严重类召回率：1.00 ✅（高危缺陷零漏报）
  - 致命类召回率：1.00 ✅（高危缺陷零漏报）
  - 轻微类召回率：0.17 ⚠️（少数类漏检严重，但业务影响小）

### 关键发现
- 逻辑回归在严重/致命类上表现完美，但对轻微类（仅6个测试样本）几乎无法识别
- 原因：轻微类样本太少，且与一般类特征边界模糊
- 改进方向：class_weight="balanced" 或增加轻微类样本
- 业务结论：模型可用——高危缺陷零漏报，轻微缺陷漏报可接受

### 遇到的问题
- 暂无